In [1]:
import os
import sys

# Standard interactive replacement for the 'parent directory' hack
parent_dir = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.append(parent_dir)

import pandas as pd
import numpy as np
from tqdm import tqdm

from PPRCalculator import PPRCalculator
from utils import species_groups, remove_cycles, mat_from_np, get_seq2name, get_DC, SpeciesGroup

In [2]:
GE_14 = 0.3059360730593607
GE_16 = 0.29213483146067415
GE_17 = 1.0

EE_14 = 0.48304227
EE_16 = 0.7744968
EE_17 = 0.2676308

DC = pd.DataFrame({
    19: [0, 0, 0, 0],
    17: [0, 0, 0, 0],
    16: [0.3, 0.5, 0, 0.2],
    14: [0.9, 0, 0.05, 0.05]
}, index=[19, 17, 16, 14]).T

DC

,19,17,16,14
19,0.0,0.0,0.00,0.00
17,0.0,0.0,0.00,0.00
16,0.3,0.5,0.00,0.20
14,0.9,0.0,0.05,0.05


In [3]:
# SPPR_16
(0.5/(GE_16) + 0.3/(GE_16) + 0.2/(GE_16) * 0.9/(GE_14*EE_14))

6.907859400014105

In [4]:
(0.5/(GE_16*EE_16) + 0.3/(GE_16*EE_16) + 0.2/(GE_16*EE_16) * 0.9/(GE_14*EE_14)) * EE_16

6.907859400014106

In [5]:
# SPPR_14
0.9/(GE_14) + (0.05/(GE_14))*(0.3/(GE_16*EE_16) + 0.5/(GE_16*EE_16))

3.519655969482357

In [6]:
(0.9/(GE_14*EE_14) + (0.05/(GE_14*EE_14))*((0.3/(GE_16*EE_16)) + 0.5/(GE_16*EE_16))) * EE_14

3.519655969482357

In [7]:
3.519656

3.519656

# EwE

In [8]:
# model_number = 496
# model_number = 240
# model_number = 473
# model_number = 240
model_number = 7
model_number = 40
self = PPRCalculator(model_number)

In [9]:
import igraph as ig
DC = self.get_DC(DET_as_PP=True)
nodes_tuple = tuple(DC.index)
num_nodes = len(nodes_tuple)
DC_vals = DC.values
adj_matrix = (DC_vals != 0).astype(int)
g = ig.Graph.Adjacency(adj_matrix.tolist(), mode="directed")
# 3. Identify terminals
terminal_mask = (DC_vals == 0).all(axis=1)
terminal_indices = np.where(terminal_mask)[0].tolist()
terminal_nodes = [nodes_tuple[i] for i in terminal_indices]
num_terminals = len(terminal_indices)

# Map terminal indices to their column index in the final 2D array
term_idx_to_col = {t_idx: col for col, t_idx in enumerate(terminal_indices)}
paths_dict = {n: {s: [] for s in terminal_nodes} for n in nodes_tuple}
for start_idx, start_node in tqdm(enumerate(nodes_tuple), desc="Outer Loop", total=len(nodes_tuple)):
    # Edge case: start node is already a terminal
    if start_idx in terminal_indices:
        paths_dict[start_node][start_node].append([start_node])
        continue
    
    # Fetch all simple paths at C-speed (returns list of integer lists)
    # paths = g.get_all_simple_paths(start_idx, to=terminal_indices)
    paths = self._get_paths_with_safety_valve(g, start_idx, terminal_indices, max_paths=1_000_000)

Outer Loop:  32%|███▎      | 13/40 [00:00<00:01, 17.90it/s]

3 1 1
3 3 1
4 1 2
4 3 3
4 5 3
5 1 1
5 3 1
6 1 2
6 3 2
7 1 1
7 3 1
8 1 1
8 3 1
9 1 2
9 3 8
9 5 8
10 1 1
10 3 9
10 5 9
11 1 1
11 3 24
11 5 26
11 7 26
12 1 1
12 3 92
12 5 3765
12 7 49639
12 9 324431
12 11 1055772
13 1 0
13 3 17
13 5 743
13 7 14702
13 9 133924


Outer Loop:  38%|███▊      | 15/40 [00:01<00:03,  6.57it/s]

13 11 607986
13 13 1383924
14 1 0
14 3 12
14 5 13
14 7 13
15 1 1
15 3 201
15 5 10472
15 7 261792


Outer Loop:  40%|████      | 16/40 [00:03<00:07,  3.01it/s]

...

37 9 590112
37 11 3051610
38 1 0
38 3 267
38 5 14101
38 7 321211


Outer Loop:  98%|█████████▊| 39/40 [00:29<00:01,  1.59s/it]

38 9 4430020
39 1 1
39 3 120
39 5 7184
39 7 166890


Outer Loop: 100%|██████████| 40/40 [00:31<00:00,  1.28it/s]

39 9 2292967


In [ ]:
sppr, A, paths = self.SPPR_EwE(TE_option='TE', use_EE=True, return_paths=True, silent=False, timeout=100)
sppr, A = PPRCalculator.rename_results([sppr, A], self.seq2name)
print(self.get_PPR(sppr, only_inner=True).rename(columns=self.seq2name).sum(axis=1))
print(self.get_PPR(sppr, only_inner=False).rename(columns=self.seq2name).sum(axis=1))
print(self.get_PPR(sppr).rename(columns=self.seq2name))
print(f'balanced: {self.is_sppr_balanced(sppr)}')
sppr.sum(axis=1).round(4).rename(index=self.name2seq).sort_index().rename(index=self.seq2name)
# sppr

In [ ]:
DC = self.get_DC(DET_as_PP=True)
TE = self.get_TE(TE_option='TE', as_matrix=True,  DET_values=1)
A = (DC / TE).fillna(0)
EE = self.EE

In [ ]:
GE_14 = 0.3059360730593607
GE_16 = 0.29213483146067415
GE_17 = 1.0
GE_19 = 1.0

EE_14 = 0.48304227
EE_16 = 0.7744968
EE_17 = 0.2676308
EE_19 = 1

TE = pd.Series({
    14: GE_14*EE_14,
    16: GE_16*EE_16,
    17: GE_17*EE_17,
    19: GE_19*EE_19,
}).T.to_frame()

EE = pd.Series({
    14: EE_14,
    16: EE_16,
    17: EE_17,
    19: EE_19,
}).T.to_frame()

DC = pd.DataFrame({
    19: [0, 0, 0, 0],
    17: [0, 0, 0, 0],
    16: [0.3, 0.5, 0, 0.2],
    14: [0.9, 0, 0.05, 0.05]
}, index=[19, 17, 16, 14]).T

A = DC.div(TE[0], axis=0)